# Spherical Harmonics with pixell


*Written by the ACT Collaboration*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/simonsobs/pixell_tutorials/blob/master/pixell_spherical_harmonics.ipynb)

---

This notebook, and the accompanying notebooks included in this set, are designed to help users who are new to working with [`pixell`](https://github.com/simonsobs/pixell/) get started with the package. As a set these notebooks will guide users through examples of how to read in and display maps, how to perform spherical harmonic transform and calculate simple spectra, how to transform the maps and how to study point sources in the maps.

The `pixell` library allows users to load,
manipulate and analyze maps stored in rectangular pixelization. It is
mainly targeted for use with maps of the sky (e.g. CMB intensity and polarization maps, stacks of 21 cm intensity maps, binned galaxy positions or shear) in cylindrical projection.

In this notebook we'll look at how to get the *alms* of a map with `pixell`. We'll also demonstrate how to filter the *alms* and how to project the *alms* back to map space. Lastly we'll demonstrate how the *alms* can be used with [`healpy`](https://healpy.readthedocs.io/en/latest/index.html), a python wrapper of `healpix`.

In [ ]:
# Download the data needed for the notebook
!wget https://phy-act1.princeton.edu/public/zatkins/act_planck_dr5.01_s08s18_AA_f150_night_map_d56.fits

In [ ]:
# Install neccesary packages
!pip install pixell

In [ ]:
# Import packages
from pixell import curvedsky, enmap, enplot
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt

**Reading maps from disk**

For more details on how to use maps in pixell take a look at the map manipulation notebook!

In [ ]:
## This map has I, Q, U components which can each be accessed individually
imap = enmap.read_map("act_planck_dr5.01_s08s18_AA_f150_night_map_d56.fits")

print(imap[0].shape)

# To access the I component we can call imap[0].
# Q and U would be imap[1] and imap[2] respectively.

A spherical harmonic transform (SHT) is analagous to an FFT, but for maps that live on the sphere rather than the flat sky. Their output is typically referred to as an *alm*, which are the components in the input map in the [spherical harmonic basis](https://en.wikipedia.org/wiki/Spherical_harmonics#Spherical_harmonics_expansion). Conventionally, each *alm* is represented as a 1d vector: all of the 2d information in the map gets reshaped into a 1d list of numbers in spherical harmonic space. If the map has a polarization component (so, a shape like `(3, Ny, Nx)`), then the output shape will be `(3, nalm)`, etc.

**Converting between maps and alms**

`pixell.curvedsky` has functions that allow users to convert maps to and from alms. In the next cell we'll demonstrate how to use these.



In [ ]:
# The transformation is limited by the `lmax` factor. Increasing this will make
# the alms exact on smaller scales but will also result in slower run times
lmax = 4000

alms = curvedsky.map2alm(imap[0], lmax=lmax)

Unlike an FFT, the SHT is not fully information-preserving. Specifically, we need to select the maximum "ell" or "lmax" -- the minimum angular scale -- to calculate. Selecting a low value for the lmax will only calculate the *alm* for the larger-scale features, but will be faster, and vice versa. The size of the *alm* will reflect the specified lmax.

Let's query the alms shape. The spherical harmonics assume the input map is real to speed up calculation. Thus we expect there to be `(lmax + 1) * (lmax + 2) / 2` (why?) complex numbers:

In [ ]:
print(alms.shape)

print((lmax+1) * (lmax+2) / 2, alms.dtype)

As expected, it is a 1d representation of the map in the spherical harmonic basis (up to ell=4000). If you want to know where in this array each *l* and *m* is located, we can do it (but it is rarely necessary):

In [ ]:
# an "alm_info" object contains lots of info and methods that can help with
# SHTs!
ainfo = curvedsky.alm_info(lmax=lmax)
ind = ainfo.lm2ind(l=100, m=99)
print(ind, alms[ind])

Users can also pass an `alm_info` object into many `curvedsky` functions instead of `lmax` directly. This can be helpful if, for instance, we wish to get the returned *alm* in a "rectangular" (2d-compatible, useful for dealing with l's separately from m's) ordering instead of the default (1d, called "triangular"):

In [ ]:
ainfo_rect = curvedsky.alm_info(lmax=lmax, layout='rectangular')
alms_rect = curvedsky.map2alm(imap[0], ainfo=ainfo_rect)

print(alms_rect.shape)
print(f'expect (lmax+1)**2 = {(lmax+1)**2}')

# l's are across the columns, m's are across the rows
plt.imshow(np.log10(abs(alms_rect)).reshape(-1, (lmax+1)), origin='lower', vmin=-4, vmax=0)
plt.colorbar()
plt.xlabel('l')
plt.ylabel('m')

`alm_info` objects have other useful methods (such as transferring from one layout to another, and multiplying by l-dependent functions), and we encourage advanced users to check them out!

`pixell` also can transform an *alm* back to map space. Again, unless an `alm_info` object is explicitly passed, it is assumed that an *alm* is in the default "triangular" layout.

In [ ]:
# You can also convert the alms back to maps with `curvedsky`. This will project
# the alms onto an existing enmap so you'll also need to provide an enmap for
# the function. If it's not empty, it will be overwritten!

out = imap[0]*0 # a new ndmap with 0's and the same shape, dtype, wcs as the input
omap = curvedsky.alm2map(alms, out)

In [ ]:
# We can plot the two maps and check if they look consistent

enplot.pshow(imap[0], downgrade=4, colorbar=True, range=250)
enplot.pshow(omap, downgrade=4, colorbar=True, range=250)
enplot.pshow(omap - imap[0], downgrade=4, colorbar=True, range=250)

Note the main difference between the output and the input is "ringing" around bright point sources. This is because we specified an lmax of 4000 -- we bandlimited the input and therefore excluded smaller scale information. Point sources have signal at small scales -- we are seeing the high-frequency (small-scale) information beyond l's of 4000.

Like the FFT `map2harm` function, we can handle polarized maps automatically. Remember `imap` is polarized:

In [ ]:
print(imap.shape)

enplot.pshow(imap[0], downgrade=4, colorbar=True)
enplot.pshow(imap[1:], downgrade=4, colorbar=True, color='gray')

By default, `map2alm` will assume a polarized map with 3 components -- I, Q, and U -- and give the output *alms* in T (same as I), E, and B. This is controlled by the `spin` argument:

In [ ]:
# We can also convert the polarized maps to E and B maps using the `map2alm` function

# Spin tells `map2alm` how to treat the different components.
# 0 refers to a scalar transform for the I (T) component
# 2 refers to a spin-2 transform for the Q and U components.

alms = curvedsky.map2alm(imap, spin=[0,2], lmax=lmax) # spin=[0, 2] is default

print(alms.shape)

Likewise, by default, `alm2map` converts from T, E, and B back into I, Q and U:

In [ ]:
out = imap*0 # a new ndmap with 0's and the same shape, dtype, wcs as the input
omap = curvedsky.alm2map(alms, out) # spin=[0, 2] is default

print(omap.shape)

enplot.pshow(omap[0], downgrade=4, colorbar=True)
enplot.pshow(omap[1:], downgrade=4, colorbar=True, color='gray')

If we want to visualize the E and B maps, rather than Q and U, we first need to do `map2alm` with the default `spin=[0,2]` and then `alm2map` with a spin-0 transform:

In [ ]:
# we can then project these to map space
IEBmap = curvedsky.alm2map(alms, imap.copy(), spin=[0,0]) # specify "0" for the Q, U components now

print(IEBmap.shape)

enplot.pshow(IEBmap[0], downgrade=4, colorbar=True)
enplot.pshow(IEBmap[1:], downgrade=4, colorbar=True, color='gray')

As expected, we see signal-dominated E modes, but B is consistent with noise!

**Filtering the maps**

Given some filter `fl` you can also filter alms using the `curvedsky.almxfl` function.

In [ ]:
# Lets start by creating a filter
fl = np.ones(lmax+1) # l=0 is one of the ells!
fl[:200] = 0         # cut out large scales

# Let's plot the filter.
plt.plot(fl)
plt.ylabel("Filter value")
plt.xlabel("ell")
plt.show()

This filter will set large scale modes to 0. The filter itself could be any function of ell that you choose, this one was just chosen to demonstrate the process.

In [ ]:
# Let's apply the filter
alms = curvedsky.map2alm(imap[0], lmax=lmax)
filtered_alms = curvedsky.almxfl(alms, fl)

In [ ]:
# We can also try filtering out the small scales for comparison
fl = np.ones(lmax + 1)
fl[200:] = 0

filtered_alms_small_scales = curvedsky.almxfl(alms, fl)

In [ ]:
# We can then project the filtered alms to map space and take a look at the resulting map
filtered_map = curvedsky.alm2map(filtered_alms, imap[0].copy())
filtered_map_small_scales = curvedsky.alm2map(filtered_alms_small_scales, imap[0].copy())
enplot.pshow(filtered_map, downgrade=4, colorbar=True)
enplot.pshow(filtered_map_small_scales, downgrade=4, colorbar=True)

Here we can see that in the first map the large scale modes are no longer visible as they've been filtered out. On the other hand, the second map shows just the large scale modes as we've removed the small scale modes with the second filter. `almxfl` can also accept functions of ell instead of a filter array.

**Calculate a power spectrum**

Just like in the FFT case, we can calculate the power in the maps as a function of angular scale. Very conveniently, because the *alms* are naturally in a basis where angular scale is directly indexed by `l`, we don't need any intermediate and inexact "binning" function like for FFTs. Instead we just use `curvedsky.alm2cl`.

We do still want to apodize the map edges to avoid discontinuities at the map edge:

In [ ]:
# get an apodized mask, multiply the map before doing map2alm
taper_mask = enmap.apod(enmap.ones(imap[0].shape, imap[0].wcs), width=100)
alms_taper = curvedsky.map2alm(taper_mask * imap, lmax=lmax)

# get the correction factor that accounts for the power lost due to only observing a
# fraction of the sky
# enmap.pixsizemap is a map of all the physical pixel areas in steradians
w2 = np.sum(taper_mask.pixsizemap() * taper_mask**2) / (4*np.pi)

# squaring and averaging over m is done by the alm2cl function
cl = curvedsky.alm2cl(alms_taper) / w2

# The shape here is refers to TT, EE, and BB spectra
print(cl.shape)

We can plot the cls:

In [ ]:
l = np.arange(cl.shape[1])

# plot
for i in range(3):
  plt.semilogy(l, cl[i] * l *(l+1) / 2 / np.pi, label=['TT', 'EE', 'BB'][i])
plt.ylim(1e-1, 1e4)
plt.legend()

By default, `alm2cl` will just get the autospectrum of each component in the *alms*. We can also pass a second, different *alm* to take a cross spectrum. Or, we can get the cross spectra between the different components of an alm like so:

In [ ]:
# squaring and averaging over m is done by the alm2cl function
# take cross spectra using numpy array broadcasting of the input shapes
clx = curvedsky.alm2cl(alms_taper[:, None], alms_taper[None, :]) / w2

print(clx.shape)

# plot
for i in range(3):
  plt.semilogy(l, clx[i, i] * l *(l+1) / 2 / np.pi, label=['TT', 'EE', 'BB'][i])
plt.ylim(1e-1, 1e4)
plt.legend()
plt.show()

for i in range(3):
  for j in range(i + 1, 3):
    plt.plot(l, clx[i, j] * l *(l+1) / 2 / np.pi, label='TEB'[i] + 'TEB'[j])
plt.legend()
plt.show()

A real cosmological analysis would need to use a *mode coupling matrix*, not the simple `w2` factor shown here!

**Using Healpy with alms**

Other than the coordinate system, *alms* don't know anything about the map pixelization that produced them. So they are a way of "translating" between different pixelization schemes (but with a the bandlimit set by `lmax`!). This means that once we have an *alm*, we can project it back into a map that uses the `healpix` pixelization using `healpy`:

We can also project the alms to a `healpix` map and plot it with `healpy`. Note that pixell also has specific functions designed for reprojecting to `healpix` which are documented in the  reprojection notebook.

In [ ]:
hp_map = hp.alm2map(alms_taper[0], 4096)

# Because we're only using a small patch of the sky we use cartview to plot
# and pass `lonra` and `latra` to select the area of the sky we wish to show.
hp.cartview(hp_map, min = -300, max=300, lonra =  [-10,45], latra =  [-8,5])

One thing to note is that these reprojections wont automatically set the masked pixells to `hp.UNSEEN`. If you wish to mask data we recommend you project a mask to `healpix` and then assign `hp.UNSEEN` to masked regions based on the reprojected mask.

**Further reading and other examples**

The ACT DR4 and DR5 notebooks, which are publicly [available on github](https://github.com/ACTCollaboration/DR4_DR5_Notebooks), include a more realistic example of how to compute power spectra in [Notebook 7](https://github.com/ACTCollaboration/DR4_DR5_Notebooks/blob/master/Notebooks/Section_7_power_spectra_part_1.ipynb).

For more examples of how to use `pixell` please see the full set of notebooks available on github.